In [ ]:

import nltk
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')

!pip install spacy
!pip install medspacy

import string
import spacy
from nltk.corpus import stopwords
from spacy.lang.en import STOP_WORDS
import medspacy
from medspacy.ner import TargetRule
from medspacy.visualization import visualize_ent
import re

#import requests
from bs4 import BeautifulSoup
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from string import digits, punctuation

# load the tokenizer, tagger, ...from spacy
# Option 2: Load from existing model
# spacy_nlp = spacy.load("en_core_sci_md", disable={"ner", "parser"})
!python -m spacy download en_core_web_sm
spacy_nlp = spacy.load("en_core_web_sm", disable={"ner", "parser"})
spacy_nlp = medspacy.load(spacy_nlp)
pd.set_option("display.max_rows", 2000)

import os
from google.colab import drive
drive.mount('/content/drive')

from concurrent.futures import ProcessPoolExecutor

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 53.6 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
C_KWIC_WINDOW_SIZE = 8
location = '/content/drive/MyDrive/AI/'

# Initialize the target matcher for lung cancer
target_matcher = spacy_nlp.get_pipe("medspacy_target_matcher")

# Define target rules for lung cancer-related concepts
target_rules = [
    # Lung Cancer-related problems
    TargetRule("Lung Cancer", "DISORDER"),
    TargetRule("Non-Small Cell Lung Cancer", "DISORDER_SUBTYPE"),
    TargetRule("Small Cell Lung Cancer", "DISORDER_SUBTYPE"),
    TargetRule("Stage IV Lung Cancer", "DISORDER_STAGE"),
    TargetRule("Adenocarcinoma", "DISORDER_SUBTYPE"),

    # Associated disorders
    TargetRule("Chronic Obstructive Pulmonary Disease", "ASSOCIATED_DISORDER"),
    TargetRule("Pulmonary Fibrosis", "ASSOCIATED_DISORDER"),
    TargetRule("Lung Infection", "ASSOCIATED_DISORDER"),
    TargetRule("Pleural Effusion", "ASSOCIATED_DISORDER"),
    TargetRule("Metastatic Lung Cancer", "ASSOCIATED_DISORDER"),

    # Medications related to lung cancer treatment
    TargetRule("Cisplatin", "MEDICATION"),
    TargetRule("Carboplatin", "MEDICATION"),
    TargetRule("Erlotinib", "MEDICATION"),
    TargetRule("Pembrolizumab", "MEDICATION"),
    TargetRule("Nivolumab", "MEDICATION"),
    TargetRule("Atezolizumab", "MEDICATION"),
    TargetRule("Docetaxel", "MEDICATION"),
    TargetRule("Vinorelbine", "MEDICATION"),

    # Other treatment-related terms
    TargetRule("Chemotherapy", "TREATMENT"),
    TargetRule("Radiation Therapy", "TREATMENT"),
    TargetRule("Targeted Therapy", "TREATMENT"),
    TargetRule("Immunotherapy", "TREATMENT"),
    TargetRule("Surgery", "TREATMENT"),

    # Lung Cancer Symptoms
    TargetRule("Cough", "SYMPTOM"),
    TargetRule("Shortness of Breath", "SYMPTOM"),
    TargetRule("Chest Pain", "SYMPTOM"),
    TargetRule("Fatigue", "SYMPTOM"),
    TargetRule("Weight Loss", "SYMPTOM"),
    TargetRule("Hoarseness", "SYMPTOM"),
    TargetRule("Wheezing", "SYMPTOM"),
    TargetRule("Swelling in Neck or Face", "SYMPTOM"),

    # Professionals and Doctors
    TargetRule("Oncologist", "PROFESSIONAL"),
    TargetRule("Pulmonologist", "PROFESSIONAL"),
    TargetRule("Radiologist", "PROFESSIONAL"),
    TargetRule("Thoracic Surgeon", "PROFESSIONAL"),
    TargetRule("Medical Oncologist", "PROFESSIONAL"),

    # Diagnosis-related terms
    TargetRule("Lung Cancer Diagnosis", "DIAGNOSIS"),
    TargetRule("Imaging Tests", "DIAGNOSIS"),
    TargetRule("Biopsy", "DIAGNOSIS"),
    TargetRule("CT Scan", "DIAGNOSTIC_TEST"),
    TargetRule("MRI", "DIAGNOSTIC_TEST"),
    TargetRule("PET Scan", "DIAGNOSTIC_TEST"),

    # Education strategies
    TargetRule("Patient Education", "EDUCATION_STRATEGY"),
    TargetRule("Support Groups", "EDUCATION_STRATEGY"),
    TargetRule("Informational Resources", "EDUCATION_STRATEGY"),

    # Support Resources
    TargetRule("Lung Cancer Support Group", "SUPPORT_RESOURCE"),
    TargetRule("Online Support Groups", "SUPPORT_RESOURCE"),
    TargetRule("Cancer Care Services", "SUPPORT_RESOURCE"),

    # Lifestyle Changes
    TargetRule("Quit Smoking", "LIFESTYLE_CHANGE"),
    TargetRule("Healthy Diet", "LIFESTYLE_CHANGE"),
    TargetRule("Regular Exercise", "LIFESTYLE_CHANGE"),
    TargetRule("Stress Management", "LIFESTYLE_CHANGE"),
    TargetRule("Avoiding Alcohol", "LIFESTYLE_CHANGE"),
]

# Add the defined target rules to the matcher
target_matcher.add(target_rules)
# -   -   -   -   -   -   -   -   -   -   -   -   -   -   -   -   -
# Define a function for bespoke cleanup of single words.
# -   -   -   -   -   -   -   -   -   -   -   -   -   -   -   -   -
def remove_single_char_words(text):
  return ' '.join([word for word in text.split() if len(word) > 1])
# -   -   -   -   -   -   -   -   -   -   -   -   -   -   -   -   -
# Define a function to do some text clearning .. only if necessary
# -   -   -   -   -   -   -   -   -   -   -   -   -   -   -   -   -
def f_cleanCorpus(corpus):
  cleaned_corpus = []

  for doc in corpus:
    doc = doc.translate(str.maketrans('', '', punctuation))
    doc = doc.translate(str.maketrans('', '', digits))
    doc = remove_single_char_words(doc)
    cleaned_corpus.append(doc)

  return cleaned_corpus
# -   -   -   -   -   -   -   -   -   -   -   -   -   -   -   -   -
# Define a function to load the documents in to a single corpus
# -   -   -   -   -   -   -   -   -   -   -   -   -   -   -   -   -
def loadCorpus(files):
  # empty corpus of abstracts
  corpus = []
  cols =[]

  # create the content corpus
  for fn in files:
    cols.append(fn)
    f = open(location+fn, 'r')
    _text = f.read()
    corpus.append(_text)

  # clean the corpus
  clean_corpus = f_cleanCorpus(corpus)

  return clean_corpus
# -   -   -   -   -   -   -   -   -   -   -   -   -   -   -   -   -
# Define a function for kwic modified
# -   -   -   -   -   -   -   -   -   -   -   -   -   -   -   -   -
def kwic_ngrams(text, phrase, window_size=C_KWIC_WINDOW_SIZE):
  # Normalize and split the text into words
  words = re.findall(r'\w+', text)
  phrase_tokens = phrase.split()
  phrase_len = len(phrase_tokens)
  kwic_lines = []

  # Iterate through the words to find matches with the phrase
  for i in range(len(words) - phrase_len + 1):
    # Check if the current slice matches the phrase
    if words[i:i + phrase_len] == phrase_tokens:
      # Determine the start and end of the window
      start = max(0, i - window_size)
      end = min(len(words), i + phrase_len + window_size)

      # Extract the context line
      context = ' '.join(words[start:end])

      # Append the KWIC line (phrase and its context)
      kwic_lines.append([phrase, context])

  return kwic_lines
# -   -   -   -   -   -   -   -   -   -   -   -   -   -   -   -   -
# Define a function for POS tagging
# -   -   -   -   -   -   -   -   -   -   -   -   -   -   -   -   -
def kwicPOS(text):
  nlp_ = spacy_nlp(text)
  pos_list = []

  for token in nlp_:
    pos_list.append([token.text, token.pos_])

  return pos_list
# -   -   -   -   -   -   -   -   -   -   -   -   -   -   -   -   -
# Define a function knowledge extration as Triples (possibly)
# -   -   -   -   -   -   -   -   -   -   -   -   -   -   -   -   -
def lookForKnowledge(df):
  triples = []
  entity_set = set(df['entity'].values)  # Set of all recognized entities

  # Iterate over each row in the dataframe
  for _, row in df.iterrows():
    s = row['entity']
    p = ''
    o = ''
    p_semphore = 0
    o_semphore = 0

    # Extract predicate and object based on POS tags
    for list_pos in row['pos_tags']:
      word = list_pos[0]

      # If the object hasn't been set and the predicate has been found, start building the object
      if o_semphore != 1 and p_semphore == 1 and word in entity_set:
        o = o + ' ' + word
        o_semphore = 1

      # If the POS tag is a VERB, set it as the predicate
      if list_pos[1] == 'VERB' and p_semphore != 1:
        p = word
        p_semphore = 1

    # Make sure subject, predicate, and object are distinct
    if s and p and o and s != p and p != o and s != o:
      triples.append([s, p, o])

  # Remove triples where the subject and object are the same
  filtered_triples = []

  for triple in triples:
    subject = triple[0].strip()
    object_ = triple[2].strip()

    if subject != object_:
      filtered_triples.append(triple)

  # Return the filtered list of triples
  return pd.DataFrame(filtered_triples, columns=["Subject", "Predicate", "Object"])


def unify_all_triples(triples):
  unified_triples = []

  while triples:
    current_triple = triples.pop(0)
    merged = False

    for i, other_triple in enumerate(triples):
      # Try to unify the current triple with each of the remaining triples
      result = unify_triples(current_triple, other_triple)

      if result != "failure":
        # Replace the current and other triples with the unified result
        triples[i] = result
        merged = True

        break

    if not merged:
      unified_triples.append(current_triple)

  # Return the unified list of triples
  return unified_triples


def unify_triples(triple1, triple2, theta=None):
  if theta is None:
    theta = {}

  subject1, predicate1, object1 = triple1
  subject2, predicate2, object2 = triple2

  # Check if subject and object match (after applying any substitutions in the theta map)
  subject1 = apply_substitution(subject1, theta)
  subject2 = apply_substitution(subject2, theta)
  object1 = apply_substitution(object1, theta)
  object2 = apply_substitution(object2, theta)

  # Unify subject and object
  theta = unify(subject1, subject2, theta)

  if theta == "failure":
    return "failure"

  theta = unify(object1, object2, theta)
  if theta == "failure":
    return "failure"

  # If the subjects and objects match, we can unify predicates
  predicate1 = apply_substitution(predicate1, theta)
  predicate2 = apply_substitution(predicate2, theta)

  # Unify predicates (which is just comparing them)
  if predicate1 != predicate2:
    return "failure"

  # Return the unified triple (subject, predicate, object)
  return apply_substitution(triple1, theta)


def apply_substitution(term, theta):
  # Check if term is a list and apply substitution to each element
  if isinstance(term, list):
    return [apply_substitution(item, theta) for item in term]

  # Check if term is a string or a variable in theta
  if isinstance(term, str) and term in theta:
    return theta[term]

  return term


def unify(x, y, theta=None):
  if theta is None:
    theta = {}

  if theta == "failure":
    return "failure"
  elif x == y:
    return theta
  elif is_variable(x):
    return unify_var(x, y, theta)
  elif is_variable(y):
    return unify_var(y, x, theta)
  elif is_compound(x) and is_compound(y) and op(x) == op(y):
    return unify(args(x), args(y), theta)
  elif is_list(x) and is_list(y):
    return unify(rest(x), rest(y), unify(first(x), first(y), theta))
  else:
    return "failure"


def unify_var(var, x, theta):
  if var in theta:
    return unify(theta[var], x, theta)
  elif x in theta:
    return unify(var, theta[x], theta)
  elif occur_check(var, x):
    return "failure"
  else:
    theta[var] = x

    return theta


def is_variable(x):
  return isinstance(x, str) and x.islower()


def is_compound(x):
  return isinstance(x, tuple) and len(x) > 1


def op(x):
  return x[0] if is_compound(x) else None


def args(x):
  return x[1:] if is_compound(x) else []


def is_list(x):
  return isinstance(x, list)


def first(x):
  return x[0] if is_list(x) and x else None


def rest(x):
  return x[1:] if is_list(x) else []


def occur_check(var, x):
  if var == x:
    return True
  elif is_compound(x):
    return any(occur_check(var, arg) for arg in args(x))
  elif is_list(x):
    return any(occur_check(var, elem) for elem in x)
  else:
    return False


# Main function for processing the corpus
def main():
    list_of_files = os.listdir(location)
    corpus = loadCorpus(list_of_files)

    df_kwic_result = pd.DataFrame(columns=['entity', 'kwic_text', 'pos_tags'])
    list_entities = []
    kwic_data = []

    for document in corpus:
        nlp_doc = spacy_nlp(document)

        for ent in nlp_doc.ents:
            if ent.label_ in ['DISORDER', 'DISORDER_SUBTYPE', 'ASSOCIATED_DISORDER', 'MEDICATION', 'TREATMENT', 'SYMPTOM', 'PROFESSIONAL', 'DIAGNOSIS', 'EDUCATION_STRATEGY', 'SUPPORT_RESOURCE', 'LIFESTYLE_CHANGE']:
                list_entities.append([ent.text, ent.start_char, ent.end_char, ent.label_])
                kwic_result = kwic_ngrams(document, ent.text)

                for kwic in kwic_result:
                    pos_tags = kwicPOS(kwic[1])
                    kwic_data.append({'entity': kwic[0], 'kwic_text': kwic[1], 'pos_tags': pos_tags})

    df_kwic_result = pd.DataFrame(kwic_data)
    knowledge_triples = lookForKnowledge(df_kwic_result)
    df_entities = pd.DataFrame(list_entities, columns=['entity', 'start', 'end', 'label'])

    return df_entities, df_kwic_result, knowledge_triples

# Run the main function
entities, kwic_results, triples = main()

# Output the results
print("\nEntities Found:")
entities = entities.drop_duplicates(subset=['entity', 'label'])
print(entities)
print("\nKWIC Results:")
print(kwic_results)
triples = triples.drop_duplicates(subset=['Subject', 'Object'])
print("\nExtracted Knowledge Triples:")
print(triples)


Entities Found:
                                    entity  start    end                label
0                              Lung Cancer     31     42             DISORDER
1                              lung cancer    638    649             DISORDER
3                   small cell lung cancer    926    948     DISORDER_SUBTYPE
6                              Lung cancer    371    382             DISORDER
14                            healthy diet   1013   1025     LIFESTYLE_CHANGE
32                                 surgery   4197   4204            TREATMENT
36                          Adenocarcinoma   4580   4594     DISORDER_SUBTYPE
39                          adenocarcinoma   5700   5714     DISORDER_SUBTYPE
51                            chemotherapy   7646   7658            TREATMENT
53                           immunotherapy   7887   7900            TREATMENT
88                   Lung Cancer Diagnosis   4515   4536            DIAGNOSIS
91                   lung cancer diagnosis   56